# 🌿 Plant Disease Detection — Training Pipeline

GPU-accelerated training and evaluation on the PlantVillage dataset
(38 classes, ~87K images).

**Deep Learning Models:**
Custom CNN · EfficientNet-B0 · EfficientNet-V2-S · ResNet-50 · ConvNeXt-Tiny

**Classical ML:**
SVM (on CNN feature embeddings)

**Features:**
AMP · Two-Phase Training (Frozen → Fine-tuned) · Early Stopping ·
TensorBoard · Checkpointing · Grad-CAM · Albumentations


## 1. Environment Verification


In [ ]:
import sys
import os
import time
import json
import warnings
import datetime
import random
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import models
from PIL import Image
from tqdm.auto import tqdm
import joblib

warnings.filterwarnings('ignore')

# ── Environment Info ───────────────────────────────────────────────
print(f"Python version     : {sys.version}")
print(f"PyTorch version    : {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version       : {torch.version.cuda}")
    print(f"cuDNN version      : {torch.backends.cudnn.version()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"  GPU {i}: {props.name} ({vram_gb:.1f} GB VRAM)")
    torch.backends.cudnn.benchmark = True
else:
    print("[!] No GPU detected. Training will be slow.")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {DEVICE}")


## 2. Central Configuration


In [ ]:
@dataclass
class Config:
    """Central configuration for the entire pipeline."""
    # ── Paths ─────────────────────────────────────────────────────
    project_root: Path = Path('.').resolve()
    dataset_dir: Path = field(default=None)
    train_dir: Path = field(default=None)
    valid_dir: Path = field(default=None)
    test_dir: Path = field(default=None)
    models_dir: Path = field(default=None)
    logs_dir: Path = field(default=None)

    # ── Data ──────────────────────────────────────────────────────
    img_size: int = 224
    batch_size: int = 32
    num_workers: int = 4
    num_classes: int = 38
    pin_memory: bool = True
    persistent_workers: bool = True

    # ── Training (End-to-End) ─────────────────────────────────────────
    epochs: int = 15
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    seed: int = 42

    # ── Fine-tuning (Disabled) ────────────────────────────────────────
    finetune_epochs: int = 0
    finetune_lr: float = 1e-5
    finetune_weight_decay: float = 1e-5

    # ── Model ─────────────────────────────────────────────────────
    models_to_train: list = field(default_factory=lambda: [
        'efficientnet_b0', 'efficientnet_v2_s', 'resnet50', 'convnext_tiny'
    ])
    pretrained: bool = True
    freeze_backbone: bool = False

    # ── Optimizer & Scheduler ─────────────────────────────────────
    optimizer: str = 'adamw'
    gradient_clip: float = 1.0

    # ── AMP ──────────────────────────────────────────────────────
    use_amp: bool = True

    # ── Early Stopping ────────────────────────────────────────────
    patience: int = 7
    min_delta: float = 0.001

    # ── Fast Verify (for testing) ─────────────────────────────────
    fast_verify: bool = False

    def __post_init__(self):
        self.dataset_dir = self.project_root / 'dataset'
        self.train_dir = self.dataset_dir / 'train'
        self.valid_dir = self.dataset_dir / 'valid'
        self.test_dir = self.dataset_dir / 'test' / 'test'
        self.models_dir = self.project_root / 'models'
        self.logs_dir = self.project_root / 'runs'
        self.models_dir.mkdir(exist_ok=True)
        self.logs_dir.mkdir(exist_ok=True)
        if sys.platform == 'win32':
            self.num_workers = 0
            self.persistent_workers = False
        if self.fast_verify:
            self.epochs = 1
            self.finetune_epochs = 1
            self.batch_size = 16

# ── Create config ──────────────────────────────────────────────────
cfg = Config(
    fast_verify=os.environ.get('FAST_VERIFY', 'False').lower() in ('true', '1', 't')
)

if cfg.fast_verify:
    print("[!] FAST_VERIFY mode enabled — minimal epochs, small batches")

# ── Reproducibility ────────────────────────────────────────────────
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(cfg.seed)

print(f"\nConfiguration:")
for k, v in asdict(cfg).items():
    print(f"  {k}: {v}")


## 3. Dataset Validation


In [ ]:
def validate_dataset(cfg):
    """Verify dataset integrity: directories, class counts, corrupt images."""
    print("=" * 60)
    print("  Dataset Validation")
    print("=" * 60)

    for name, path in [('Train', cfg.train_dir), ('Valid', cfg.valid_dir), ('Test', cfg.test_dir)]:
        exists = path.exists()
        print(f"  {name:6s}: {path}  (exists: {exists})")
        assert exists or name == 'Test', f"{name} directory not found: {path}"

    class_dirs = sorted([d for d in cfg.train_dir.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_dirs]
    print(f"\n  Number of classes: {len(class_names)}")
    assert len(class_names) == cfg.num_classes, \
        f"Expected {cfg.num_classes} classes, found {len(class_names)}"

    summary = []
    for split_name, split_dir in [('train', cfg.train_dir), ('valid', cfg.valid_dir)]:
        for cls_dir in sorted(split_dir.iterdir()):
            if not cls_dir.is_dir():
                continue
            img_count = len(list(cls_dir.glob('*')))
            summary.append({'split': split_name, 'class': cls_dir.name, 'count': img_count})

    df_summary = pd.DataFrame(summary)
    train_total = df_summary[df_summary['split'] == 'train']['count'].sum()
    valid_total = df_summary[df_summary['split'] == 'valid']['count'].sum()
    print(f"  Training images  : {train_total:,}")
    print(f"  Validation images: {valid_total:,}")
    print(f"\n  Dataset validation passed! ✅")
    return class_names, df_summary

class_names, dataset_summary = validate_dataset(cfg)


## 4. Exploratory Data Analysis


In [ ]:
train_counts = dataset_summary[dataset_summary['split'] == 'train'].set_index('class')['count']

fig, ax = plt.subplots(figsize=(14, 10))
colors = plt.colormaps['viridis'](np.linspace(0.2, 0.9, len(train_counts)))
train_counts.sort_values().plot(kind='barh', ax=ax, color=colors)
ax.set_title('Training Images per Class', fontsize=14)
ax.set_xlabel('Number of Images')
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.show()

print(f"Mean images/class : {train_counts.mean():.0f}")
print(f"Std images/class  : {train_counts.std():.0f}")
print(f"Min images/class  : {train_counts.min()} ({train_counts.idxmin()})")
print(f"Max images/class  : {train_counts.max()} ({train_counts.idxmax()})")
print(f"Imbalance ratio   : {train_counts.max() / train_counts.min():.2f}x")


## 5. Data Augmentation


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def get_train_transforms(img_size: int) -> A.Compose:
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=10, p=0.3),
        A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.3),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transforms(img_size: int) -> A.Compose:
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

train_transforms = get_train_transforms(cfg.img_size)
val_transforms = get_val_transforms(cfg.img_size)
print("Train transforms:", train_transforms)
print("Val transforms:", val_transforms)


## 6. PyTorch Dataset & DataLoaders


In [ ]:
class PlantDiseaseDataset(Dataset):
    """PyTorch Dataset for plant disease images with Albumentations support."""

    def __init__(self, root_dir: Path, class_names: list, transform=None):
        self.transform = transform
        self.class_to_idx = {name: i for i, name in enumerate(class_names)}
        self.samples = []
        for cls_name in class_names:
            cls_dir = root_dir / cls_name
            if not cls_dir.exists():
                continue
            for img_path in cls_dir.iterdir():
                if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp'):
                    self.samples.append((img_path, self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = np.array(Image.open(img_path).convert('RGB'))
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label

# ── Create datasets ────────────────────────────────────────────────
train_dataset = PlantDiseaseDataset(cfg.train_dir, class_names, train_transforms)
val_dataset = PlantDiseaseDataset(cfg.valid_dir, class_names, val_transforms)

print(f"Full training samples  : {len(train_dataset):,}")
print(f"Full validation samples: {len(val_dataset):,}")

if cfg.fast_verify:
    train_indices = []
    t_counts = {}
    for idx, (_, label) in enumerate(train_dataset.samples):
        t_counts[label] = t_counts.get(label, 0) + 1
        if t_counts[label] <= 4:
            train_indices.append(idx)
    train_dataset = torch.utils.data.Subset(train_dataset, train_indices)

    val_indices = []
    v_counts = {}
    for idx, (_, label) in enumerate(val_dataset.samples):
        v_counts[label] = v_counts.get(label, 0) + 1
        if v_counts[label] <= 2:
            val_indices.append(idx)
    val_dataset = torch.utils.data.Subset(val_dataset, val_indices)
    print(f"[!] FAST_VERIFY subset: {len(train_dataset)} train, {len(val_dataset)} val ({len(t_counts)} classes)")

worker_args = dict(
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory if (torch.cuda.is_available() and cfg.num_workers > 0) else False,
    persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, **worker_args)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, **worker_args)

print(f"Training batches  : {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")

class_names_path = cfg.models_dir / 'class_names.json'
with open(class_names_path, 'w') as f:
    json.dump(class_names, f, indent=2)
print(f"\nSaved class names to {class_names_path}")


## 7. Deep Learning Model Zoo

Five selected architectures — quality over quantity.


In [ ]:
class CustomCNN(nn.Module):
    """Custom 5-block CNN with BatchNorm and Global Average Pooling."""
    def __init__(self, num_classes: int = 38):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(name: str, num_classes: int = 38,
                pretrained: bool = True,
                freeze_backbone: bool = True) -> nn.Module:
    """Factory function to build any supported model."""
    weights = 'DEFAULT' if pretrained else None

    if name == 'custom_cnn':
        return CustomCNN(num_classes)

    elif name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=weights)
        if freeze_backbone:
            for param in model.features.parameters():
                param.requires_grad = False
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
        return model

    elif name == 'efficientnet_v2_s':
        model = models.efficientnet_v2_s(weights=weights)
        if freeze_backbone:
            for param in model.features.parameters():
                param.requires_grad = False
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
        return model

    elif name == 'resnet50':
        model = models.resnet50(weights=weights)
        if freeze_backbone:
            for name_p, param in model.named_parameters():
                if 'fc' not in name_p:
                    param.requires_grad = False
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(in_features, num_classes),
        )
        return model

    elif name == 'convnext_tiny':
        model = models.convnext_tiny(weights=weights)
        if freeze_backbone:
            for param in model.features.parameters():
                param.requires_grad = False
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
        return model

    else:
        raise ValueError(
            f"Unknown model: {name}. Supported: custom_cnn, efficientnet_b0, "
            f"efficientnet_v2_s, resnet50, convnext_tiny"
        )

# ── Model Factory ─────────────────────────────────────────────────
print("Models scheduled for training:")
for m in cfg.models_to_train:
    print(f" - {m}")


## 8. Training Pipeline


In [ ]:
class EarlyStopping:
    """Early stopping to halt training when validation loss stops improving."""
    def __init__(self, patience: int = 7, min_delta: float = 0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False

    def __call__(self, val_loss: float) -> bool:
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
        return self.should_stop

    def reset(self):
        self.counter = 0
        self.best_loss = None
        self.should_stop = False


def train_one_epoch(model, loader, criterion, optimizer, scaler, device,
                    gradient_clip=1.0, use_amp=True):
    """Train for one epoch with AMP and gradient clipping."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type='cuda', enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix(loss=loss.item(), acc=correct / total)

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device, use_amp=True):
    """Validate the model on the validation set."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc='Validation', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        with autocast(device_type='cuda', enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


In [ ]:
def run_training(model, train_loader, val_loader, cfg, device, model_name):
    """End-to-end training loop."""
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    params = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = optim.AdamW(params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)
    scaler = GradScaler(enabled=cfg.use_amp)
    early_stopping = EarlyStopping(patience=cfg.patience, min_delta=cfg.min_delta)

    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    writer = SummaryWriter(str(cfg.logs_dir / f'{model_name}_{timestamp}'))

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    best_val_acc = 0.0
    save_path = cfg.models_dir / f'{model_name}_best.pth'

    print(f"\n{'=' * 60}")
    print(f"  Training: {model_name}")
    print(f"  Epochs: {cfg.epochs} | LR: {cfg.learning_rate}")
    print(f"{'=' * 60}\n")

    start_time = time.time()

    for epoch in range(cfg.epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler,
            device, cfg.gradient_clip, cfg.use_amp
        )
        val_loss, val_acc = validate(model, val_loader, criterion, device, cfg.use_amp)

        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        writer.add_scalars('Loss', {'train': train_loss, 'val': val_loss}, epoch)
        writer.add_scalars('Accuracy', {'train': train_acc, 'val': val_acc}, epoch)
        writer.add_scalar('Learning Rate', current_lr, epoch)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        print(f"  Epoch {epoch + 1:3d}/{cfg.epochs} | "
              f"Train: {train_acc:.4f} | Val: {val_acc:.4f} | "
              f"Loss: {val_loss:.4f} | LR: {current_lr:.2e}", end='')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            checkpoint = {
                'epoch': epoch + 1,
                'model_name': model_name,
                'model_state_dict': model.state_dict(),
                'best_val_acc': best_val_acc,
                'config': asdict(cfg),
                'class_names': class_names,
            }
            torch.save(checkpoint, save_path)
            print(f" ★ Saved (best: {best_val_acc:.4f})")
        else:
            print()

        if early_stopping(val_loss):
            print(f"\n  Early stopping triggered at epoch {epoch + 1}")
            break

    training_time = time.time() - start_time
    writer.close()
    return history, training_time, best_val_acc


In [ ]:
# ── Execute Training for all models ────────────────────────────────
results = {}
for model_name in cfg.models_to_train:
    print(f"\n\nInitializing {model_name}...")
    model = build_model(model_name, cfg.num_classes, cfg.pretrained, cfg.freeze_backbone)
    model = model.to(DEVICE)

    history, t_time, best_acc = run_training(model, train_loader, val_loader, cfg, DEVICE, model_name)
    results[model_name] = best_acc

    # Free up GPU memory before next model
    del model
    torch.cuda.empty_cache()

print("\n\nTraining Summary:")
for m, acc in results.items():
    print(f"  - {m}: {acc:.4f}")

# ── Plot learning curves ───────────────────────────────────────────
from evaluation.evaluate_metrics import plot_training_history
plot_training_history(history, model_name=model_name)


## 9. Train All Deep Learning Models

Sequential training of all 5 architectures with two-phase strategy.


In [ ]:
ALL_DL_MODELS = [
    'custom_cnn', 'efficientnet_b0', 'efficientnet_v2_s',
    'resnet50', 'convnext_tiny',
]

all_dl_results = {}

for model_name in ALL_DL_MODELS:
    print(f"\n\n{'#' * 70}")
    print(f"  MODEL: {model_name}")
    print(f"{'#' * 70}")

    m = build_model(model_name, cfg.num_classes, cfg.pretrained, cfg.freeze_backbone)
    m = m.to(DEVICE)

    total_params = sum(p.numel() for p in m.parameters())
    trainable_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Parameters: {total_params:,} total, {trainable_params:,} trainable")

    hist, t_time, best = run_training(m, train_loader, val_loader, cfg, DEVICE, model_name)

    all_dl_results[model_name] = {
        'history': hist,
        'training_time': t_time,
        'best_val_acc': best,
        'total_params': total_params,
        'trainable_params': trainable_params,
    }

    plot_training_history(hist, model_name=model_name)

    del m
    torch.cuda.empty_cache()

print("\n\n✅ All deep learning models trained!")


## 10. Comprehensive Evaluation


In [ ]:
from evaluation.evaluate_metrics import (
    compute_metrics, plot_confusion_matrix,
    print_classification_report, plot_roc_curve_multiclass,
    plot_pr_curve_multiclass,
)

@torch.no_grad()
def get_predictions(model, loader, device, use_amp=True):
    """Get all predictions and ground truth from a dataloader."""
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in tqdm(loader, desc='Predicting', leave=False):
        images = images.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=use_amp):
            outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.numpy())
        all_probs.append(probs)

    return np.concatenate(all_preds), np.concatenate(all_labels), np.concatenate(all_probs)


def evaluate_dl_model(model_name, val_loader, cfg, device):
    """Load a saved model and run full evaluation."""
    model_path = cfg.models_dir / f'{model_name}_best.pth'
    if not model_path.exists():
        print(f"  [SKIP] {model_name}: no saved model found")
        return None

    m = build_model(model_name, cfg.num_classes, pretrained=False, freeze_backbone=False)
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    m.load_state_dict(checkpoint['model_state_dict'])
    m = m.to(device)

    start_time = time.time()
    y_pred, y_true, y_proba = get_predictions(m, val_loader, device, cfg.use_amp)
    inference_time = time.time() - start_time

    metrics = compute_metrics(y_true, y_pred, y_proba, model_name)
    metrics['inference_time'] = inference_time
    metrics['training_time'] = all_dl_results.get(model_name, {}).get('training_time', 0)
    metrics['total_params'] = sum(p.numel() for p in m.parameters())
    metrics['model_size_mb'] = model_path.stat().st_size / (1024 * 1024)

    print(f"\n{'=' * 60}")
    print(f"  {model_name}")
    print(f"{'=' * 60}")
    print(f"  Accuracy   : {metrics['accuracy']:.4f}")
    print(f"  Precision  : {metrics['precision']:.4f}")
    print(f"  Recall     : {metrics['recall']:.4f}")
    print(f"  F1 Score   : {metrics['f1']:.4f}")
    print(f"  Top-5 Acc  : {metrics.get('top5_accuracy', 'N/A')}")

    plot_confusion_matrix(y_true, y_pred, class_names, title=f'{model_name} — Confusion Matrix')
    print_classification_report(y_true, y_pred, class_names)

    del m
    torch.cuda.empty_cache()
    return metrics

all_eval_results = []
for model_name in ALL_DL_MODELS:
    result = evaluate_dl_model(model_name, val_loader, cfg, DEVICE)
    if result:
        all_eval_results.append(result)

print(f"\nEvaluated {len(all_eval_results)} deep learning models.")


## 11. Explainability — Grad-CAM


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def get_target_layer(model, model_name):
    if model_name == 'custom_cnn':
        return [model.features[-3]]
    elif model_name in ('efficientnet_b0', 'efficientnet_v2_s'):
        return [model.features[-1]]
    elif model_name == 'resnet50':
        return [model.layer4[-1]]
    elif model_name == 'convnext_tiny':
        return [model.features[-1]]
    else:
        return [list(model.children())[-2]]

def visualize_gradcam(model, model_name, dataset, class_names, device, num_samples=8):
    model.eval()
    target_layers = get_target_layer(model, model_name)
    cam = GradCAM(model=model, target_layers=target_layers)

    fig, axes = plt.subplots(2, num_samples, figsize=(num_samples * 3, 6))
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))

    for i, idx in enumerate(indices):
        image_tensor, label = dataset[idx]
        input_tensor = image_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
        pred = output.argmax(dim=1).item()
        conf = torch.softmax(output, dim=1).max().item()

        targets = [ClassifierOutputTarget(pred)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

        img_np = image_tensor.permute(1, 2, 0).numpy()
        img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)

        axes[0, i].imshow(img_np)
        color = 'green' if pred == label else 'red'
        axes[0, i].set_title(
            f"True: {class_names[label].split('___')[-1][:15]}\n"
            f"Pred: {class_names[pred].split('___')[-1][:15]}\n"
            f"Conf: {conf:.1%}", fontsize=7, color=color
        )
        axes[0, i].axis('off')

        cam_image = show_cam_on_image(img_np.astype(np.float32), grayscale_cam, use_rgb=True)
        axes[1, i].imshow(cam_image)
        axes[1, i].set_title('Grad-CAM', fontsize=8)
        axes[1, i].axis('off')

    plt.suptitle(f'Grad-CAM — {model_name}', fontsize=14)
    plt.tight_layout()
    plt.show()

# ── Run Grad-CAM on the best model ─────────────────────────────────
best_model_name = max(all_dl_results, key=lambda k: all_dl_results[k]['best_val_acc'])
model_path = cfg.models_dir / f'{best_model_name}_best.pth'
if model_path.exists():
    gradcam_model = build_model(best_model_name, cfg.num_classes, pretrained=False, freeze_backbone=False)
    ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    gradcam_model.load_state_dict(ckpt['model_state_dict'])
    gradcam_model = gradcam_model.to(DEVICE)
    visualize_gradcam(gradcam_model, best_model_name, val_dataset, class_names, DEVICE, num_samples=8)
    del gradcam_model
    torch.cuda.empty_cache()


## 12. Classical ML — SVM on CNN Features

A single SVM classifier trained on CNN feature embeddings from the
best performing deep learning model.


In [ ]:
from sklearn.svm import SVC

# ── CNN Feature Extraction ─────────────────────────────────────────
@torch.no_grad()
def extract_cnn_features(model, model_name, loader, device):
    """Extract features from the penultimate layer of a CNN."""
    if model_name == 'custom_cnn':
        feature_model = nn.Sequential(model.features, nn.Flatten())
    elif model_name in ('efficientnet_b0', 'efficientnet_v2_s', 'convnext_tiny'):
        feature_model = nn.Sequential(model.features, nn.AdaptiveAvgPool2d(1), nn.Flatten())
    elif model_name == 'resnet50':
        feature_model = nn.Sequential(*list(model.children())[:-1], nn.Flatten())
    else:
        feature_model = nn.Sequential(*list(model.children())[:-1], nn.Flatten())

    feature_model = feature_model.to(device).eval()

    features_list, labels_list = [], []
    for images, labels in tqdm(loader, desc='Extracting features', leave=False):
        images = images.to(device, non_blocking=True)
        feats = feature_model(images).cpu().numpy()
        features_list.append(feats)
        labels_list.append(labels.numpy())

    return np.vstack(features_list), np.concatenate(labels_list)

# ── Use best DL model for feature extraction ───────────────────────
best_model_name = max(all_dl_results, key=lambda k: all_dl_results[k]['best_val_acc'])
best_dl_path = cfg.models_dir / f'{best_model_name}_best.pth'

feat_model = build_model(best_model_name, cfg.num_classes, pretrained=False, freeze_backbone=False)
ckpt = torch.load(best_dl_path, map_location=DEVICE, weights_only=False)
feat_model.load_state_dict(ckpt['model_state_dict'])
feat_model = feat_model.to(DEVICE)

print(f"Extracting features using: {best_model_name}")
X_train_cnn, y_train = extract_cnn_features(feat_model, best_model_name, train_loader, DEVICE)
X_val_cnn, y_val = extract_cnn_features(feat_model, best_model_name, val_loader, DEVICE)
print(f"Feature shape: {X_train_cnn.shape}")

del feat_model
torch.cuda.empty_cache()

# ── Train SVM ──────────────────────────────────────────────────────
print("\nTraining SVM...")
svm_model = SVC(kernel='rbf', probability=True, random_state=cfg.seed, C=10)
start = time.time()
svm_model.fit(X_train_cnn, y_train)
svm_train_time = time.time() - start

y_proba = svm_model.predict_proba(X_val_cnn)
y_pred = y_proba.argmax(axis=1)
svm_metrics = compute_metrics(y_val, y_pred, y_proba, 'SVM')
svm_metrics['inference_time'] = 0
svm_metrics['training_time'] = svm_train_time
svm_metrics['total_params'] = 0

save_path = cfg.models_dir / 'svm_model.pkl'
joblib.dump(svm_model, str(save_path))
svm_metrics['model_size_mb'] = save_path.stat().st_size / (1024 * 1024)

print(f"  SVM — Accuracy: {svm_metrics['accuracy']:.4f} — Time: {svm_train_time:.1f}s")
print(f"  Saved: {save_path.name} ({svm_metrics['model_size_mb']:.1f} MB)")


## 13. Model Comparison & Ranking


In [ ]:
from evaluation.evaluate_metrics import plot_model_comparison

all_results = all_eval_results + [svm_metrics]
results_df = pd.DataFrame(all_results)

display_cols = ['model', 'accuracy', 'precision', 'recall', 'f1',
                'top5_accuracy', 'training_time', 'model_size_mb']
available_cols = [c for c in display_cols if c in results_df.columns]
display_df = results_df[available_cols].sort_values('accuracy', ascending=False)

print("\n" + "=" * 100)
print("  MODEL COMPARISON TABLE")
print("=" * 100)
print(display_df.to_string(index=False))

best_model = display_df.iloc[0]
print(f"\n🏆 Best Model: {best_model['model']} (Accuracy: {best_model['accuracy']:.4f})")

results_df.to_csv(cfg.models_dir / 'comparison.csv', index=False)

metrics_json = {
    'best_model': best_model['model'],
    'best_accuracy': float(best_model['accuracy']),
    'timestamp': datetime.datetime.now().isoformat(),
    'models': results_df.to_dict(orient='records'),
}
with open(cfg.models_dir / 'metrics.json', 'w') as f:
    json.dump(metrics_json, f, indent=2, default=str)

with open(cfg.models_dir / 'training_config.json', 'w') as f:
    json.dump(asdict(cfg), f, indent=2, default=str)

plot_model_comparison(all_results)

fig, ax = plt.subplots(figsize=(12, 5))
sorted_df = results_df.sort_values('accuracy', ascending=True)
colors = plt.colormaps['viridis'](np.linspace(0.2, 0.8, len(sorted_df)))
bars = ax.barh(sorted_df['model'], sorted_df['accuracy'], color=colors)
ax.set_xlabel('Accuracy')
ax.set_title('Model Accuracy Comparison')
ax.set_xlim(0, 1.05)
for bar, acc in zip(bars, sorted_df['accuracy']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{acc:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\n✅ All models trained, evaluated, and compared!")
